# P5 DEV inference with HF + PEFT

Use Kaggle GPU T4 x2 and Internet On. This notebook only reads the 32 exported p5-v1 prompt JSON files and the private adapter input dataset. It never reads repository corpora, benchmark data, or test data.

Before running, retrieve the immutable base revision, NF4 compute dtype, and double-quant setting from the P6 training notebook/logs. This notebook stops on placeholders. It runs the no-adapter baseline and both adapters sequentially on the same 4-bit NF4 base, with greedy decoding, seed 42, max_new_tokens 2048, context limit 8192.

In [ ]:
from pathlib import Path
import runpy

# Frozen P5 loading settings. The base revision is an explicit approximation, not training provenance.
P5_CONFIG = {
    'repo_commit': '9792aad',
    'base_model': 'Qwen/Qwen2.5-Coder-7B-Instruct',
    'base_revision': 'c03e6d358207e414f1eca0bb1891e29f1db0e242',  # approximation, not recovered from training
    'base_revision_status': 'approximation',
    'base_revision_source': 'Hugging Face history, latest commit before 2026-11-20 (2025-01-12); not training provenance',
    'nf4_compute_dtype': 'float16',  # explicit in rank32 training code; assumed identical for qlora
    'nf4_double_quant': True,  # explicit in rank32 training code; assumed identical for qlora
    'model_dtype': 'float16',
    'device_map': {'': 0},  # single GPU; no sharding
    'gpu_devices_used': [0],
    'prompt_dir': '/kaggle/input/p5-dev-prompts/prompts',
    'qlora_adapter_dir': '/kaggle/input/p5-adapters/qlora',
    'rank32_adapter_dir': '/kaggle/input/p5-adapters/rank32',
    'output_dir': '/kaggle/working/p5-results',
}
RUNNER = Path('/kaggle/input/p5-dev-prompts/p5_kaggle_inference.py')
if not RUNNER.is_file():
    raise FileNotFoundError(f'Missing P5 inference runner: {RUNNER}')

In [ ]:
# Run the packaged inference function with the explicit configuration above.
import runpy
namespace = runpy.run_path(str(RUNNER))
namespace['run_p5_inference'](P5_CONFIG)

# Download /kaggle/working/p5-results/p5-dev-generations.jsonl and manifest.json after completion.